In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import textwrap

# 1. Configuración de conexión Cloud a RDS (SQL Server)
USER_RDS = ''
PASS_RDS = ''
ENDPOINT_RDS = ''
DB_NAME = ''

cadena_conexion = f"mssql+pyodbc://{USER_RDS}:{PASS_RDS}@{ENDPOINT_RDS}:1433/{DB_NAME}?driver=ODBC+Driver+17+for+SQL+Server"
engine = create_engine(cadena_conexion)

# 2. Extracción de los datos base
print("Extrayendo datos de housing_analytics.kc_house...")
query_base = "SELECT * FROM housing_analytics.kc_house;"
df = pd.read_sql(query_base, engine)

# 3. Preparación: Extraer el año de la columna 'date'
# La columna date viene como '20141013T000000', tomamos los primeros 4 caracteres
df['year'] = df['date'].str[:4]

# --- TRANSFORMACIÓN 1 ---
# Promedio de precio, tamaño, número de habitaciones por casa por zipcode por año.
print("\nGenerando Tabla 1: Agregados por Zipcode y Año...")
tabla_1 = df.groupby(['zipcode', 'year']).agg(
    avg_price=('price', 'mean'),
    avg_sqft=('sqft_living', 'mean'),
    avg_bedrooms=('bedrooms', 'mean')
).reset_index()

# Formatear según preferencias
pd.options.display.float_format = '{:,.2f}'.format
print(tabla_1.head())

# --- TRANSFORMACIÓN 2 ---
# Precio promedio de casa por año y por zipcode.
# (Nota: La instrucción 2 es muy similar a la 1, solo enfocada en precio.
# La estructuraremos ligeramente diferente para que sea una vista útil).
print("\nGenerando Tabla 2: Evolución de Precio Medio por Zipcode/Año...")
tabla_2 = df.groupby(['year', 'zipcode'])['price'].mean().reset_index(name='avg_price')
print(tabla_2.head())

# 4. Carga: Guardar las nuevas tablas en el RDS (Data Warehouse)
print("\nCargando nuevas tablas al esquema housing_analytics en RDS...")
try:
    # Según protocolo: se guardan en el esquema housing_analytics
    tabla_1.to_sql('agg_metrics_zip_year', con=engine, schema='housing_analytics', if_exists='replace', index=False)
    tabla_2.to_sql('agg_price_year_zip', con=engine, schema='housing_analytics', if_exists='replace', index=False)
    print("--- ETL AVANZADO COMPLETADO EXITOSAMENTE ---")
except Exception as e:
    print(f"Error durante la carga: {e}")

Extrayendo datos de housing_analytics.kc_house...

Generando Tabla 1: Agregados por Zipcode y Año...
   zipcode  year  avg_price  avg_sqft  avg_bedrooms
0    98001  2014 275,250.87  1,889.02          3.36
1    98001  2015 292,434.49  1,925.65          3.44
2    98002  2014 234,215.78  1,640.14          3.37
3    98002  2015 234,418.51  1,603.31          3.24
4    98003  2014 292,311.72  1,930.50          3.34

Generando Tabla 2: Evolución de Precio Medio por Zipcode/Año...
   year  zipcode    avg_price
0  2014    98001   275,250.87
1  2014    98002   234,215.78
2  2014    98003   292,311.72
3  2014    98004 1,295,630.17
4  2014    98005   803,386.54

Cargando nuevas tablas al esquema housing_analytics en RDS...
--- ETL AVANZADO COMPLETADO EXITOSAMENTE ---


In [6]:
import os
import warnings
from fpdf import FPDF

warnings.filterwarnings("ignore", category=DeprecationWarning)

class PDFReport(FPDF):
    def header(self):
        self.set_font('helvetica', 'B', 15)
        self.set_fill_color(34, 49, 63)
        self.set_text_color(255, 255, 255)
        self.cell(0, 15, ' Práctica M55: ETL Avanzado y Cloud BI Dashboard', border=0, 
                  new_x="LMARGIN", new_y="NEXT", align='L', fill=True)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('helvetica', 'I', 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f'Autor: Patrick Salvador Hernández Arias - EBAC | Página {self.page_no()}', align='C')

def generar_reporte_m55():
    ruta_base = r'C:\Users\patri\Downloads'
    
    pdf = PDFReport()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # PORTADA
    pdf.add_page()
    pdf.set_font('helvetica', 'B', 16)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 10, "Reporte Técnico de Arquitectura de Datos", new_x="LMARGIN", new_y="NEXT")
    
    pdf.set_font('helvetica', '', 11)
    pdf.set_text_color(80, 80, 80)
    texto_intro = (
        "El presente documento evidencia la construcción de un pipeline de datos completo (End-to-End). "
        "Se demuestra la extracción de datos en la nube, la transformación automatizada simulando un "
        "trabajo de AWS Glue mediante Python, la persistencia optimizada utilizando Vistas en un Data Warehouse "
        "en AWS RDS (SQL Server), y finalmente, el consumo directo en Looker Studio respetando estándares IBCS."
    )
    pdf.multi_cell(0, 7, texto_intro)
    pdf.ln(10)

    # DIAPOSITIVAS
    diapositivas = [
        ("d1.png", "1. Automatización ETL (Simulación AWS Glue)", 
         "El código en Python extrae la data base del RDS, genera las agrupaciones requeridas por zona y año, y carga las nuevas tablas de vuelta a la nube. Este script está diseñado para ejecutarse diariamente de forma automatizada."),
        
        ("d2.png", "2. Optimización en Capa de Datos (SQL Server Views)", 
         "Para reducir latencia y costos de transferencia (AWS Egress), se crearon Vistas SQL. Esto permite procesar campos concatenados y pre-filtrar columnas en el servidor, entregando a Looker Studio únicamente la información necesaria."),
        
        ("d3.png", "3. Visión General del Mercado (Gráfico de Burbujas)", 
         "El dashboard principal despliega el volumen y valor geográfico de las propiedades. El uso de burbujas cruza múltiples dimensiones (precio, ubicación y densidad), aportando un nivel analítico superior al mapa tradicional."),
        
        ("d4.png", "4. Relación de Características (ETL 1 - Tabla Agregada)", 
         "Lectura directa de la primera vista generada por el ETL. El gráfico evalúa la relación entre las métricas habitacionales y el precio, gobernado por tres filtros dinámicos sincronizados con la nube."),
        
        ("d5.png", "5. Tendencia Histórica Geográfica (ETL 2 - Evolución)", 
         "El panel consolida la tendencia anual del valor inmobiliario segmentado por códigos postales. La automatización del ETL asegura que las variaciones del mercado se reflejen en este componente sin intervención manual.")
    ]

    imagenes_encontradas = 0

    for archivo, titulo, insight in diapositivas:
        ruta_completa = os.path.join(ruta_base, archivo)
        
        if os.path.exists(ruta_completa):
            pdf.add_page()
            pdf.set_font('helvetica', 'B', 13)
            pdf.set_text_color(44, 62, 80)
            pdf.cell(0, 10, titulo, new_x="LMARGIN", new_y="NEXT")
            
            pdf.set_font('helvetica', '', 11)
            pdf.set_text_color(80, 80, 80)
            pdf.multi_cell(0, 6, insight)
            pdf.ln(4)
            
            ancho_util = pdf.epw
            pdf.image(ruta_completa, x=15, w=ancho_util)
            imagenes_encontradas += 1
        else:
            print(f"[-] Advertencia: No se encontró la imagen '{archivo}'.")

    # ENLACES
    pdf.add_page()
    pdf.set_font('helvetica', 'B', 14)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 10, "Cierre de Arquitectura y Enlaces de Acceso", new_x="LMARGIN", new_y="NEXT")
    
    pdf.set_font('helvetica', '', 11)
    pdf.set_text_color(50, 50, 50)
    pdf.multi_cell(0, 7, "Para interactuar con la presentación ejecutiva y el tablero dinámico alojado en Looker Studio, por favor acceda a los siguientes enlaces públicos de los recursos en la nube:")
    pdf.ln(5)
    
    pdf.set_font('helvetica', 'B', 11)
    pdf.cell(0, 8, "> Enlace a Google Slides (Wireframes y Presentación):", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('helvetica', '', 11)
    pdf.set_text_color(41, 128, 185) 
    pdf.cell(0, 8, "[ https://datastudio.google.com/reporting/c6f139e3-3b5b-4d14-a7c1-a00409ddf802 ]", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)

    if imagenes_encontradas > 0:
        ruta_salida = os.path.join(ruta_base, 'Practica_M55_Pipeline_Patrick.pdf')
        pdf.output(ruta_salida)
        print("\n--- PROCESO COMPLETADO ---")
        print(f"[+] Se procesaron {imagenes_encontradas}/5 imágenes correctamente.")
        print(f"[+] El PDF se ha guardado en: {ruta_salida}")
    else:
        print("\n[!] ERROR: No se encontró ninguna imagen. Revisa los nombres de d1.png a d5.png.")

generar_reporte_m55()


--- PROCESO COMPLETADO ---
[+] Se procesaron 5/5 imágenes correctamente.
[+] El PDF se ha guardado en: C:\Users\patri\Downloads\Practica_M55_Pipeline_Patrick.pdf
